In [2]:
import json
import pymongo
import os
import sys
from bson import ObjectId
from datetime import datetime
from bson.json_util import dumps
from bson.json_util import loads
from bson import json_util
from bson import Binary
from bson import Code
from bson import DBRef
from bson import SON
from bson import Timestamp
from bson import MinKey
from bson import MaxKey
from bson import Decimal128
from bson import json_util
from bson import json_util
from bson import json_util
from bson import json_util
import pandas as pd
from pymongo import MongoClient
from pymongo import InsertOne
from pymongo import UpdateOne
from pymongo import DeleteOne
from pymongo import ReplaceOne
from pymongo import IndexModel
from pymongo import ASCENDING
from pymongo import DESCENDING
from pymongo.errors import OperationFailure
from pymongo.errors import ConfigurationError
from pymongo.errors import PyMongoError
from pymongo.errors import ServerSelectionTimeoutError
from pymongo.errors import DuplicateKeyError
from pymongo.errors import WriteError
from pymongo.errors import WriteConcernError
from pymongo.errors import BulkWriteError
from pymongo.errors import InvalidName
from pymongo.errors import CollectionInvalid
from pymongo.errors import InvalidURI
from pymongo.errors import NetworkTimeout
from pymongo.errors import AutoReconnect
from pymongo.errors import DocumentTooLarge
from pymongo.errors import ExecutionTimeout
from pymongo.errors import CursorNotFound
from pymongo.errors import InvalidOperation
from pymongo.errors import ConfigurationError


In [3]:
# Conexão MongoDB
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["BD_Producao_Artistica"]
collection = db["equipe_raw"]

# Processamento por linha (Eficiente em memória)
with open('equipe.jsonl', 'r', encoding='utf-8') as f:
    batch = []
    for line in f:
        batch.append(json.loads(line))
        if len(batch) == 1000:  # Insere em lotes para performance
            collection.insert_many(batch)
            batch = []
    if batch:
        collection.insert_many(batch)

In [5]:
# Configurações de Conexão (Ajuste se o seu Docker usar outra porta ou autenticação)
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["BD_Producao_Artistica"]

def carregar_jsonl_otimizado(nome_arquivo, nome_colecao, batch_size=5000):
    path_completo = os.path.join(r'C:\Users\User\Documents\FATESG\Banco de dados não relacional\Trabalho Prático – ETL e Armazenamento em Banco de Dados Não Relacional\Data', nome_arquivo)
    collection = db[nome_colecao]
    
    print(f"Iniciando carga de: {nome_arquivo}...")
    
    with open(path_completo, 'r', encoding='utf-8') as f:
        batch = []
        contador = 0
        for line in f:
            try:
                # Remove espaços em branco e ignora linhas vazias
                line = line.strip()
                if not line:
                    continue
                
                batch.append(json.loads(line))
                contador += 1
                
                # Quando o lote atinge o tamanho definido, insere no banco
                if len(batch) >= batch_size:
                    collection.insert_many(batch)
                    batch = []
                    print(f"Inseridos {contador} registros em {nome_colecao}...")
            except Exception as e:
                print(f"Erro na linha: {e}")
        
        # Insere o restante que sobrou no último lote
        if batch:
            collection.insert_many(batch)
            print(f"Carga finalizada: {contador} registros totais em {nome_colecao}.")

# Execução para os dois arquivos
carregar_jsonl_otimizado('pessoa.jsonl', 'pessoa_raw')
carregar_jsonl_otimizado('producao.jsonl', 'producao_raw')

Iniciando carga de: pessoa.jsonl...
Inseridos 5000 registros em pessoa_raw...
Inseridos 10000 registros em pessoa_raw...
Inseridos 15000 registros em pessoa_raw...
Inseridos 20000 registros em pessoa_raw...
Inseridos 25000 registros em pessoa_raw...
Inseridos 30000 registros em pessoa_raw...
Inseridos 35000 registros em pessoa_raw...
Inseridos 40000 registros em pessoa_raw...
Inseridos 45000 registros em pessoa_raw...
Inseridos 50000 registros em pessoa_raw...
Inseridos 55000 registros em pessoa_raw...
Inseridos 60000 registros em pessoa_raw...
Inseridos 65000 registros em pessoa_raw...
Inseridos 70000 registros em pessoa_raw...
Inseridos 75000 registros em pessoa_raw...
Inseridos 80000 registros em pessoa_raw...
Inseridos 85000 registros em pessoa_raw...
Inseridos 90000 registros em pessoa_raw...
Inseridos 95000 registros em pessoa_raw...
Inseridos 100000 registros em pessoa_raw...
Inseridos 105000 registros em pessoa_raw...
Inseridos 110000 registros em pessoa_raw...
Inseridos 115000

In [6]:
import pymongo

client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["BD_Producao_Artistica"]

def tratar_dados():
    print("Iniciando Tratamento da Etapa 3...")

    # 1. Tratamento de Produções (Anos e Tipos)
    producoes = db.producao_raw.find()
    batch_prod = []
    for p in producoes:
        # Limpeza do Ano: Se for "0", vira None
        ano = p.get('ano')
        p['ano'] = int(ano) if str(ano).isdigit() and int(ano) > 0 else None
        
        # Garante que ID seja int
        p['id_producao'] = int(p['id_producao'])
        batch_prod.append(p)
        
        if len(batch_prod) >= 5000:
            db.producao_clean.insert_many(batch_prod)
            batch_prod = []
            
    if batch_prod: db.producao_clean.insert_many(batch_prod)
    print("✓ Produções tratadas.")

    # 2. Tratamento de Equipe (Nomes e Papéis)
    equipes = db.equipe_raw.find()
    batch_eq = []
    for e in equipes:
        # Correção de Erro de Digitação (conforme identificado na amostra)
        if e.get('papel') == "Himelf":
            e['papel'] = "Himself"
        
        # Tratar papéis nulos ou vazios
        if not e.get('papel') or e['papel'] == "null":
            e['papel'] = "Unknown"

        e['id_producao'] = int(e['id_producao'])
        e['id_pessoa'] = int(e['id_pessoa'])
        
        batch_eq.append(e)
        if len(batch_eq) >= 5000:
            db.equipe_clean.insert_many(batch_eq)
            batch_eq = []

    if batch_eq: db.equipe_clean.insert_many(batch_eq)
    print("✓ Equipe tratada.")

if __name__ == "__main__":
    tratar_dados()

Iniciando Tratamento da Etapa 3...
✓ Produções tratadas.
✓ Equipe tratada.


In [7]:
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["BD_Producao_Artistica"]

def tratar_pessoas():
    print("Iniciando Tratamento da coleção Pessoa...")
    
    # 1. Busca os dados brutos
    pessoas_raw = db.pessoa_raw.find()
    batch_pessoa = []
    contador = 0

    for p in pessoas_raw:
        # Garante que o ID seja Inteiro
        try:
            p['id_pessoa'] = int(p['id_pessoa'])
        except:
            continue # Pula se o ID for inválido
            
        # Limpeza básica no nome (remover espaços extras)
        if p.get('nome'):
            p['nome'] = p['nome'].strip()
        else:
            p['nome'] = "Unknown"

        batch_pessoa.append(p)
        
        # Inserção em lotes para performance
        if len(batch_pessoa) >= 5000:
            db.pessoa_clean.insert_many(batch_pessoa)
            contador += len(batch_pessoa)
            batch_pessoa = []
            print(f"Processados {contador} registros de pessoas...")

    # Insere o restante
    if batch_pessoa:
        db.pessoa_clean.insert_many(batch_pessoa)
        contador += len(batch_pessoa)

    print(f"✓ Coleção pessoa_clean criada com {contador} registros!")

if __name__ == "__main__":
    tratar_pessoas()

Iniciando Tratamento da coleção Pessoa...
Processados 5000 registros de pessoas...
Processados 10000 registros de pessoas...
Processados 15000 registros de pessoas...
Processados 20000 registros de pessoas...
Processados 25000 registros de pessoas...
Processados 30000 registros de pessoas...
Processados 35000 registros de pessoas...
Processados 40000 registros de pessoas...
Processados 45000 registros de pessoas...
Processados 50000 registros de pessoas...
Processados 55000 registros de pessoas...
Processados 60000 registros de pessoas...
Processados 65000 registros de pessoas...
Processados 70000 registros de pessoas...
Processados 75000 registros de pessoas...
Processados 80000 registros de pessoas...
Processados 85000 registros de pessoas...
Processados 90000 registros de pessoas...
Processados 95000 registros de pessoas...
Processados 100000 registros de pessoas...
Processados 105000 registros de pessoas...
Processados 110000 registros de pessoas...
Processados 115000 registros de 

In [ ]:
import pymongo

client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["BD_Producao_Artistica"]

def criar_visao_documental():
    print("Iniciando a Etapa 4: Modelagem Documental...")
    
    # Criar um dicionário (index) de pessoas em memória para ser ultra rápido
    # (Como pessoa_clean pode ser grande, isso economiza milhares de consultas ao banco)
    print("Indexando nomes de pessoas em memória...")
    pessoas_lookup = {p['id_pessoa']: p['nome'] for p in db.pessoa_clean.find()}

    producoes = db.producao_clean.find()
    batch_final = []
    contador = 0

    for prod in producoes:
        id_prod = prod['id_producao']
        
        # Busca a equipe desta produção específica
        equipe_membros = db.equipe_clean.find({"id_producao": id_prod})
        
        lista_participantes = []
        for membro in equipe_membros:
            id_p = membro['id_pessoa']
            # Pega o nome do nosso dicionário em memória
            nome_pessoa = pessoas_lookup.get(id_p, "Nome não encontrado")
            
            lista_participantes.append({
                "id_pessoa": id_p,
                "nome": nome_pessoa,
                "papel": membro['papel']
            })
        
        # Adiciona a lista de participantes dentro do documento da produção
        prod['equipe'] = lista_participantes
        
        # Remove o _id original da coleção anterior para não dar conflito
        if '_id' in prod: del prod['_id']
        
        batch_final.append(prod)
        
        if len(batch_final) >= 1000:
            db.producoes_com_participantes.insert_many(batch_final)
            batch_final = []
            contador += 1000
            print(f"Documentos enriquecidos: {contador}...")

    if batch_final:
        db.producoes_com_participantes.insert_many(batch_final)
    
    print("✓ Etapa 4 Concluída: Coleção 'producoes_com_participantes' criada!")

if __name__ == "__main__":
    criar_visao_documental()

Iniciando a Etapa 4: Modelagem Documental...
Indexando nomes de pessoas em memória...


In [ ]:
import pymongo
from collections import defaultdict

# Conecta ao Mongo do Docker
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["BD_Producao_Artistica"]

def unificacao_ultra_rapida():
    print("--- Iniciando Etapa 4: Unificação ---")
    
    # 1. Limpa a coleção de destino para não duplicar dados
    db.producoes_com_participantes.drop()

    # 2. Carrega nomes das pessoas para a memória (Dicionário)
    print("Carregando pessoas...")
    pessoas_map = {p['id_pessoa']: p['nome'] for p in db.pessoa_clean.find()}

    # 3. Organiza a equipe por produção (Dicionário de listas)
    print("Mapeando equipes (isso pode levar 1 ou 2 minutos)...")
    equipe_por_prod = defaultdict(list)
    for eq in db.equipe_clean.find():
        id_p = eq['id_pessoa']
        equipe_por_prod[eq['id_producao']].append({
            "id_pessoa": id_p,
            "nome": pessoas_map.get(id_p, "Desconhecido"),
            "papel": eq.get('papel', 'Unknown')
        })

    # 4. Une tudo na coleção final
    print("Criando documentos finais...")
    producoes = db.producao_clean.find()
    batch = []
    
    for prod in producoes:
        prod['equipe'] = equipe_por_prod.get(prod['id_producao'], [])
        if '_id' in prod: del prod['_id'] # Remove ID antigo
        batch.append(prod)
        
        if len(batch) >= 5000:
            db.producoes_com_participantes.insert_many(batch)
            batch = []

    if batch:
        db.producoes_com_participantes.insert_many(batch)
    
    print("--- CONCLUÍDO COM SUCESSO! ---")

if __name__ == "__main__":
    unificacao_ultra_rapida()

In [7]:
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["BD_Producao_Artistica"]
col = db["producoes_com_participantes"]

def finalizar_etapa_5():
    print("--- VALIDANDO VANTAGEM DO MODELO DOCUMENTAL ---")

    # Busca flexível para ignorar variações no título
    termo_busca = "Sex and Zen II"
    doc = col.find_one({"titulo": {"$regex": termo_busca, "$options": "i"}})

    if doc:
        print(f"\n✅ SUCESSO! Filme encontrado: {doc['titulo']}")
        print(f"Ano de Lançamento: {doc.get('ano')}")
        print(f"Quantidade de pessoas na equipe: {len(doc.get('equipe', []))}")
        
        print("\n--- Detalhes da Equipe (Sem necessidade de JOIN) ---")
        for membro in doc.get('equipe', []):
            print(f"ID Pessoa: {membro['id_pessoa']} | Papel: {membro['papel']}")
            
        print("\nExplicação para o relatório:")
        print("Diferente do modelo relacional, onde precisaríamos de uma query com JOIN")
        print("entre 3 tabelas, aqui recuperamos a produção e todos os seus 12 milhões")
        print("de relacionamentos possíveis com uma única leitura de disco.")
    else:
        print("\n❌ Filme não encontrado com esse nome exato.")
        print("Sugestão: Tente buscar apenas 'Sex and Zen' ou verifique os títulos com:")
        print("db.producoes_com_participantes.find().limit(5)")

if __name__ == "__main__":
    finalizar_etapa_5()

--- VALIDANDO VANTAGEM DO MODELO DOCUMENTAL ---

✅ SUCESSO! Filme encontrado: Sex and Zen II
Ano de Lançamento: 1996
Quantidade de pessoas na equipe: 1

--- Detalhes da Equipe (Sem necessidade de JOIN) ---
ID Pessoa: 430909 | Papel: Unknown

Explicação para o relatório:
Diferente do modelo relacional, onde precisaríamos de uma query com JOIN
entre 3 tabelas, aqui recuperamos a produção e todos os seus 12 milhões
de relacionamentos possíveis com uma única leitura de disco.


In [15]:
# Configuração da Conexão
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["BD_Producao_Artistica"]
col = db["producoes_com_participantes"]

print("--- ETAPA 5: RELATÓRIO ANALÍTICO COMPLETO ---")
print("="*50)

# 1. CONSULTAS SIMPLES COM FILTRO
print("\n1.1 Produções do ano de 1996 (Amostra):")
for p in col.find({"ano": 1996}).limit(3):
    print(f" - {p.get('titulo')} (ID: {p.get('id_producao')})")

print("\n1.2 Total de produções do Tipo 1 lançadas após 2000:")
total_pos_2000 = col.count_documents({"tipo_id": 1, "ano": {"$gt": 2000}})
print(f" - Total: {total_pos_2000}")

# 2. AGREGAÇÕES (GROUP BY)
print("\n2.1 Distribuição de produções por Tipo (Agregação):")
pipeline_tipos = [
    {"$group": {"_id": "$tipo_id", "total": {"$sum": 1}}},
    {"$sort": {"total": -1}}
]
for res in col.aggregate(pipeline_tipos):
    print(f" - Tipo {res['_id']}: {res['total']} documentos")

print("\n2.2 Ranking de Anos com mais lançamentos (Top 5):")
pipeline_anos = [
    {"$match": {"ano": {"$gt": 0}}},
    {"$group": {"_id": "$ano", "qtd": {"$sum": 1}}},
    {"$sort": {"qtd": -1}},
    {"$limit": 5}
]
for res in col.aggregate(pipeline_anos):
    print(f" - Ano {res['_id']}: {res['qtd']} produções")

# 3. RANKING COMPLEXO (UNWIND)
print("\n3.1 Top 5 Papéis (Jobs) mais frequentes na base:")
# Nota: Esta operação percorre o array 'equipe' de todos os documentos
pipeline_ranking = [
    {"$unwind": "$equipe"},
    {"$group": {"_id": "$equipe.papel", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 5}
]
for res in col.aggregate(pipeline_ranking):
    print(f" - Papel: {res['_id']} | Ocorrências: {res['count']}")

# 4. VANTAGEM DO MODELO DOCUMENTAL (O GRANDE FINAL)
print("\n" + "="*50)
print("4.1 Demonstração de Documento Rico (Sem JOIN):")

# Busca flexível para garantir que encontre o filme
termo = "Sex and Zen II"
doc = col.find_one({"titulo": {"$regex": termo, "$options": "i"}})

if doc:
    print(f"Filme Encontrado: {doc.get('titulo')}")
    print(f"Ano: {doc.get('ano')}")
    print(f"Equipe embutida (Array):")
    
    equipe = doc.get('equipe', [])
    if equipe:
        for membro in equipe[:5]: # Mostra os 5 primeiros membros
            id_p = membro.get('id_pessoa', 'N/A')
            papel = membro.get('papel', 'N/A')
            print(f"  > ID Pessoa: {id_p} | Papel: {papel}")
    else:
        print("  (Nenhum membro de equipe vinculado a este título)")
else:
    print(f"Aviso: Filme '{termo}' não localizado para esta demonstração.")

print("\n--- FIM DO RELATÓRIO ETAPA 5 ---")

--- ETAPA 5: RELATÓRIO ANALÍTICO COMPLETO ---

1.1 Produções do ano de 1996 (Amostra):
 - Cousin Philmore Butts on the Prowl (ID: 95295)
 - AP Biology (ID: 25304)
 - Carmen (ID: 73311)

1.2 Total de produções do Tipo 1 lançadas após 2000:
 - Total: 66697

2.1 Distribuição de produções por Tipo (Agregação):
 - Tipo 7: 467482 documentos
 - Tipo 1: 383226 documentos
 - Tipo 3: 60594 documentos
 - Tipo 4: 56970 documentos
 - Tipo 2: 44317 documentos
 - Tipo 5: 5568 documentos
 - Tipo 6: 5459 documentos

2.2 Ranking de Anos com mais lançamentos (Top 5):
 - Ano 2005: 50254 produções
 - Ano 2004: 45013 produções
 - Ano 2003: 37200 produções
 - Ano 2006: 35150 produções
 - Ano 2002: 31180 produções

3.1 Top 5 Papéis (Jobs) mais frequentes na base:
 - Papel: Unknown | Ocorrências: 2702806
 - Papel: Himself | Ocorrências: 432269
 - Papel: Herself | Ocorrências: 82638
 - Papel: Himself - Host | Ocorrências: 40044
 - Papel: Narrator | Ocorrências: 18699

4.1 Demonstração de Documento Rico (Sem JOI

In [17]:
# 1. Instalação das bibliotecas (O comando '!' funciona aqui dentro do Jupyter)
import sys
!{sys.executable} -m pip install pandas pyarrow fastparquet

# 2. Importação das bibliotecas para a conversão
import pandas as pd
import os

print("--- CONVERSOR BIG DATA: JSONL PARA PARQUET ---")

def converter_para_parquet(arquivo_origem):
    if not os.path.exists(arquivo_origem):
        print(f"⚠️ Arquivo {arquivo_origem} não encontrado.")
        return
    
    arquivo_destino = arquivo_origem.replace('.jsonl', '.parquet')
    print(f"⏳ Processando {arquivo_origem}...")

    # Usando chunksize de 500k para não travar sua RAM com os 12 milhões de registros
    # O Parquet é um formato colunar, excelente para o volume do seu trabalho
    reader = pd.read_json(arquivo_origem, lines=True, chunksize=500000)
    
    for i, chunk in enumerate(reader):
        # Na primeira iteração cria o arquivo, nas próximas anexa (append)
        if i == 0:
            chunk.to_parquet(arquivo_destino, engine='pyarrow', index=False)
        else:
            # Para o arquivo 'equipe', ele vai passar por aqui várias vezes
            chunk.to_parquet(arquivo_destino, engine='pyarrow', index=False, append=True)
        
        if (i + 1) % 2 == 0:
            print(f"   ✅ { (i+1) * 500000 / 1000000 } Milhões de linhas processadas...")

    tamanho_mb = os.path.getsize(arquivo_destino) / (1024**2)
    print(f"🚀 {arquivo_destino} criado com sucesso! Tamanho final: {tamanho_mb:.2f} MB\n")

# Lista de arquivos para converter
arquivos = ['producao.jsonl', 'pessoa.jsonl', 'equipe.jsonl']

for f in arquivos:
    converter_para_parquet(f)

print("--- TODAS AS CONVERSÕES FINALIZADAS ---")

   ---------------------------------------- 0.0/669.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/669.4 kB ? eta -:--:--
   --------------- ------------------------ 262.1/669.4 kB ? eta -:--:--
   ---------------------------------------- 669.4/669.4 kB 2.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 2.8 MB/s eta 0:00:01
   ------------------ --------------------- 0.8/1.7 MB 2.2 MB/s eta 0:00:01
   ------------------------------ --------- 1.3/1.7 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 2.1 MB/s  0:00:00

   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fss

In [18]:
import os
import pandas as pd

# 1. Verifica onde o Python está "olhando"
print(f"O Python está procurando aqui: {os.getcwd()}")
print(f"Arquivos que ele encontrou nesta pasta: {os.listdir('.')}")

def converter_parquet_ajustado(nome_jsonl):
    # Tenta encontrar o arquivo
    if not os.path.exists(nome_jsonl):
        print(f"❌ Erro: Não achei o arquivo '{nome_jsonl}' nesta pasta.")
        return

    nome_parquet = nome_jsonl.replace('.jsonl', '.parquet')
    print(f"⏳ Convertendo {nome_jsonl}...")

    # Usando chunksize menor (250k) para garantir estabilidade nos 12 milhões
    reader = pd.read_json(nome_jsonl, lines=True, chunksize=250000)
    
    for i, chunk in enumerate(reader):
        # Para o primeiro pedaço, cria o arquivo. Para os outros, anexa.
        append_mode = False if i == 0 else True
        # Usamos fastparquet para o modo append, que é mais simples
        chunk.to_parquet(nome_parquet, engine='fastparquet', index=False, append=append_mode)
        
        if (i + 1) % 4 == 0:
            print(f"   ✅ { (i+1) * 250000 / 1000000 } Milhões de linhas processadas...")

    print(f"🚀 Sucesso! {nome_parquet} gerado.\n")

# Execute a conversão
arquivos = ['producao.jsonl', 'pessoa.jsonl', 'equipe.jsonl']
for f in arquivos:
    converter_parquet_ajustado(f)

O Python está procurando aqui: c:\Users\User\Documents\FATESG\Banco de dados não relacional\Trabalho Prático – ETL e Armazenamento em Banco de Dados Não Relacional
Arquivos que ele encontrou nesta pasta: ['Data', 'ingestao.ipynb']
❌ Erro: Não achei o arquivo 'producao.jsonl' nesta pasta.
❌ Erro: Não achei o arquivo 'pessoa.jsonl' nesta pasta.
❌ Erro: Não achei o arquivo 'equipe.jsonl' nesta pasta.


In [21]:
import pyarrow as pa
import pyarrow.parquet as pq

def converter_parquet_blindado(caminho_jsonl):
    if not os.path.exists(caminho_jsonl):
        print(f"❌ Não encontrado: {caminho_jsonl}")
        return

    caminho_parquet = caminho_jsonl.replace('.jsonl', '.parquet')
    print(f"⏳ Convertendo {caminho_jsonl} com Esquema Unificado...")

    # Forçamos o Pandas a ler colunas problemáticas como string para evitar o conflito de tipos
    # O chunksize garante que não estoura a RAM
    reader = pd.read_json(caminho_jsonl, lines=True, chunksize=500000, dtype=False)
    
    writer = None
    for i, chunk in enumerate(reader):
        # Convertemos todo o chunk para string antes de transformar em Table
        # Isso padroniza o esquema e evita o erro de 'Table schema does not match'
        chunk = chunk.astype(str) 
        table = pa.Table.from_pandas(chunk)
        
        if writer is None:
            writer = pq.ParquetWriter(caminho_parquet, table.schema, compression='snappy')
        
        writer.write_table(table)
        
        if (i + 1) % 2 == 0:
            print(f"   ✅ { (i+1) * 500000 / 1000000 } Milhões de linhas processadas...")

    if writer:
        writer.close()
    
    print(f"🚀 Sucesso! Arquivo gerado: {caminho_parquet}\n")

# Executar
arquivos = ['Data/producao.jsonl', 'Data/pessoa.jsonl', 'Data/equipe.jsonl']
for f in arquivos:
    converter_parquet_blindado(f)

⏳ Convertendo Data/producao.jsonl com Esquema Unificado...
   ✅ 1.0 Milhões de linhas processadas...
🚀 Sucesso! Arquivo gerado: Data/producao.parquet

⏳ Convertendo Data/pessoa.jsonl com Esquema Unificado...
   ✅ 1.0 Milhões de linhas processadas...
   ✅ 2.0 Milhões de linhas processadas...
🚀 Sucesso! Arquivo gerado: Data/pessoa.parquet

⏳ Convertendo Data/equipe.jsonl com Esquema Unificado...
   ✅ 1.0 Milhões de linhas processadas...
   ✅ 2.0 Milhões de linhas processadas...
   ✅ 3.0 Milhões de linhas processadas...
   ✅ 4.0 Milhões de linhas processadas...
   ✅ 5.0 Milhões de linhas processadas...
   ✅ 6.0 Milhões de linhas processadas...
   ✅ 7.0 Milhões de linhas processadas...
   ✅ 8.0 Milhões de linhas processadas...
   ✅ 9.0 Milhões de linhas processadas...
   ✅ 10.0 Milhões de linhas processadas...
   ✅ 11.0 Milhões de linhas processadas...
   ✅ 12.0 Milhões de linhas processadas...
   ✅ 13.0 Milhões de linhas processadas...
🚀 Sucesso! Arquivo gerado: Data/equipe.parquet



In [6]:
# Conexão
client = MongoClient("mongodb://localhost:27017/")
db = client["BD_Producao_Artistica"]

def carregar_parquet_no_mongo(nome_base):
    caminho_parquet = f"Data/{nome_base}.parquet"
    colecao = db[nome_base]
    
    # 1. Limpa a coleção para começar do zero
    print(f"🧹 Limpando coleção '{nome_base}'...")
    colecao.drop()
    
    print(f"📥 Carregando {caminho_parquet}...")
    
    # O Pandas lê o Parquet quase instantaneamente
    df = pd.read_parquet(caminho_parquet)
    
    # Converter para dicionário e inserir (usando insert_many para performance)
    # Convertemos em pedaços (chunks) para não estourar o limite de 16MB do buffer do Mongo
    batch_size = 50000
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size].to_dict('records')
        colecao.insert_many(batch)
        if (i + batch_size) % 1000000 == 0 or (i + batch_size) >= len(df):
            progresso = min(i + batch_size, len(df))
            print(f"   ✅ {progresso} registros inseridos...")

    print(f"⭐ {nome_base} finalizado!\n")

# Executar a carga das 3 tabelas base
list(map(carregar_parquet_no_mongo, ['producao', 'pessoa', 'equipe']))

print("--- ETAPA 2 CONCLUÍDA: DADOS BRUTOS CARREGADOS ---")

🧹 Limpando coleção 'producao'...
📥 Carregando Data/producao.parquet...
   ✅ 1000000 registros inseridos...
   ✅ 1023616 registros inseridos...
⭐ producao finalizado!

🧹 Limpando coleção 'pessoa'...
📥 Carregando Data/pessoa.parquet...
   ✅ 1000000 registros inseridos...
   ✅ 2000000 registros inseridos...
   ✅ 2062495 registros inseridos...
⭐ pessoa finalizado!

🧹 Limpando coleção 'equipe'...
📥 Carregando Data/equipe.parquet...
   ✅ 1000000 registros inseridos...
   ✅ 2000000 registros inseridos...
   ✅ 3000000 registros inseridos...
   ✅ 4000000 registros inseridos...
   ✅ 5000000 registros inseridos...
   ✅ 6000000 registros inseridos...
   ✅ 7000000 registros inseridos...
   ✅ 8000000 registros inseridos...
   ✅ 9000000 registros inseridos...
   ✅ 10000000 registros inseridos...
   ✅ 11000000 registros inseridos...
   ✅ 12000000 registros inseridos...
   ✅ 12849041 registros inseridos...
⭐ equipe finalizado!

--- ETAPA 2 CONCLUÍDA: DADOS BRUTOS CARREGADOS ---


In [7]:
print("--- ETAPA 3: TRATAMENTO DE PRODUÇÃO ---")

pipeline_producao = [
    {
        "$project": {
            "id_producao": 1,
            "titulo": {"$trim": {"input": "$titulo"}}, # Remove espaços extras
            "ano": {
                "$cond": {
                    "if": {"$and": [
                        {"$isNumber": {"$convert": {"input": "$ano", "to": "int", "onError": False}}},
                        {"$gt": [{"$convert": {"input": "$ano", "to": "int"}}, 1800]},
                        {"$lt": [{"$convert": {"input": "$ano", "to": "int"}}, 2026]}
                    ]},
                    "then": {"$convert": {"input": "$ano", "to": "int"}},
                    "else": None # Marca anos inválidos como null
                }
            },
            "tipo_id": {"$convert": {"input": "$tipo_id", "to": "int", "onError": 0}}
        }
    },
    { "$match": { "titulo": { "$ne": "" }, "ano": { "$ne": None } } }, # Descarta sem título ou sem ano
    { "$out": "producao_clean" } # Salva na nova coleção
]

db.producao.aggregate(pipeline_producao)
print("✅ Coleção 'producao_clean' criada.")

--- ETAPA 3: TRATAMENTO DE PRODUÇÃO ---
✅ Coleção 'producao_clean' criada.


In [8]:
# Verificando se os anos foram convertidos e filtrados corretamente
exemplo_clean = db.producao_clean.find_one()
total_original = db.producao.count_documents({})
total_clean = db.producao_clean.count_documents({})

print(f"Total Original: {total_original}")
print(f"Total após Limpeza: {total_clean}")
print(f"Registros descartados: {total_original - total_clean}")
print("\nExemplo de documento tratado:")
print(exemplo_clean)

Total Original: 1023616
Total após Limpeza: 845866
Registros descartados: 177750

Exemplo de documento tratado:
{'_id': ObjectId('69d65c1f728f077c2545e90a'), 'id_producao': '70625', 'titulo': "Campanile d'oro, Il", 'ano': 1955, 'tipo_id': 1}


In [5]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
db = client["BD_Producao_Artistica"]

In [6]:
print("--- ETAPA 3: TRATAMENTO DE EQUIPE ---")

pipeline_equipe = [
    {
        # Remove registros onde o papel é nulo, vazio ou a string "null"
        "$match": {
            "papel": { "$exists": True, "$ne": "", "$nin": ["null", None] },
            "id_producao": { "$ne": None },
            "id_pessoa": { "$ne": None }
        }
    },
    {
        "$project": {
            "_id": 0,
            # Garante que os IDs sejam inteiros para o Join funcionar perfeitamente
            "id_producao": {"$convert": {"input": "$id_producao", "to": "int", "onError": -1}},
            "id_pessoa": {"$convert": {"input": "$id_pessoa", "to": "int", "onError": -1}},
            "papel": { "$trim": { "input": "$papel" } }
        }
    },
    {
        # Remove o que deu erro na conversão de ID
        "$match": { "id_producao": { "$gt": 0 }, "id_pessoa": { "$gt": 0 } }
    },
    { "$out": "equipe_clean" }
]

db.equipe.aggregate(pipeline_equipe)
print("✅ Coleção 'equipe_clean' criada e tipada.")

--- ETAPA 3: TRATAMENTO DE EQUIPE ---
✅ Coleção 'equipe_clean' criada e tipada.


In [9]:
print("--- ETAPA 3: CRIAÇÃO DE ÍNDICES ---")

db.producao_clean.create_index("id_producao")
db.equipe_clean.create_index("id_producao")
db.equipe_clean.create_index("id_pessoa")
db.pessoa.create_index("id_pessoa")

print("✅ Índices criados com sucesso.")

--- ETAPA 3: CRIAÇÃO DE ÍNDICES ---
✅ Índices criados com sucesso.


In [10]:
# Verificação rápida da coleção final
total_final = db.producoes_com_participantes.count_documents({})
amostra = db.producoes_com_participantes.find_one({"equipe": {"$exists": True, "$not": {"$size": 0}}})

print(f"📊 Total de produções ricas: {total_final}")
if amostra:
    print(f"✅ Exemplo encontrado: {amostra['titulo']} com {len(amostra['equipe'])} integrantes na equipe.")
else:
    print("⚠️ Atenção: A coleção existe, mas parece não ter os dados embutidos.")

📊 Total de produções ricas: 1023616
✅ Exemplo encontrado: Campanile d'oro, Il com 9 integrantes na equipe.


In [11]:
print("--- 1.1 Produções de um ano específico (Ex: 2024) ---")
producoes_recentes = db.producoes_com_participantes.find({"ano": 2024}).limit(5)
for p in producoes_recentes:
    print(f"Título: {p['titulo']} | Ano: {p['ano']}")

print("\n--- 1.2 Produções de um tipo específico (Ex: tipo_id 1) ---")
tipo_especifico = db.producoes_com_participantes.find({"tipo_id": 1}).limit(5)
for p in tipo_especifico:
    print(f"Título: {p['titulo']} | Tipo: {p['tipo_id']}")

print("\n--- 1.3 Busca por título parcial (Case Insensitive) ---")
busca_nome = db.producoes_com_participantes.find({"titulo": {"$regex": "Matrix", "$options": "i"}}).limit(5)
for p in busca_nome:
    print(f"Título encontrado: {p['titulo']}")

--- 1.1 Produções de um ano específico (Ex: 2024) ---

--- 1.2 Produções de um tipo específico (Ex: tipo_id 1) ---
Título: Campanile d'oro, Il | Tipo: 1
Título: Caída libre | Tipo: 1
Título: Crucea de piatra | Tipo: 1
Título: Clinic, The | Tipo: 1
Título: Chauffeur, The | Tipo: 1

--- 1.3 Busca por título parcial (Case Insensitive) ---
Título encontrado: Blue Matrix
Título encontrado: Armitage III: Poly Matrix
Título encontrado: Armitage: Dual Matrix
Título encontrado: Animatrix, The
Título encontrado: Heavy Metal: Geomatrix


In [12]:
print("--- 2.1 Quantidade de produções por ano (Top 10 anos) ---")
agregacao_anos = db.producoes_com_participantes.aggregate([
    {"$group": {"_id": "$ano", "total": {"$sum": 1}}},
    {"$sort": {"total": -1}},
    {"$limit": 10}
])
for res in agregacao_anos:
    print(f"Ano: {res['_id']} | Produções: {res['total']}")

print("\n--- 2.2 Média de integrantes na equipe por tipo de produção ---")
agregacao_equipe = db.producoes_com_participantes.aggregate([
    {"$project": {"tipo_id": 1, "qtd_equipe": {"$size": "$equipe"}}},
    {"$group": {"_id": "$tipo_id", "media_equipe": {"$avg": "$qtd_equipe"}}},
    {"$sort": {"media_equipe": -1}}
])
for res in agregacao_equipe:
    print(f"Tipo: {res['_id']} | Média de Integrantes: {res['media_equipe']:.2f}")

--- 2.1 Quantidade de produções por ano (Top 10 anos) ---
Ano: None | Produções: 177749
Ano: 2005 | Produções: 50254
Ano: 2004 | Produções: 45013
Ano: 2003 | Produções: 37200
Ano: 2006 | Produções: 35150
Ano: 2002 | Produções: 31180
Ano: 2001 | Produções: 28742
Ano: 2000 | Produções: 26184
Ano: 1999 | Produções: 23634
Ano: 1998 | Produções: 21771

--- 2.2 Média de integrantes na equipe por tipo de produção ---
Tipo: 6 | Média de Integrantes: 9.28
Tipo: 4 | Média de Integrantes: 8.30
Tipo: 1 | Média de Integrantes: 8.01
Tipo: 3 | Média de Integrantes: 7.96
Tipo: 5 | Média de Integrantes: 7.75
Tipo: 7 | Média de Integrantes: 7.27
Tipo: 2 | Média de Integrantes: 7.18


In [13]:
print("--- 3.1 Ranking: Top 10 Pessoas com mais participações ---")
ranking_pessoas = db.producoes_com_participantes.aggregate([
    {"$unwind": "$equipe"}, # "Quebra" o array de equipe em documentos individuais
    {"$group": {"_id": "$equipe.id_pessoa", "contagem": {"$sum": 1}}},
    {"$sort": {"contagem": -1}},
    {"$limit": 10}
])
for rank in ranking_pessoas:
    print(f"ID Pessoa: {rank['_id']} | Participações: {rank['contagem']}")

--- 3.1 Ranking: Top 10 Pessoas com mais participações ---
ID Pessoa: 27803 | Participações: 1107
ID Pessoa: 678980 | Participações: 923
ID Pessoa: 606175 | Participações: 736
ID Pessoa: 616240 | Participações: 699
ID Pessoa: 594736 | Participações: 658
ID Pessoa: 835409 | Participações: 645
ID Pessoa: 527893 | Participações: 582
ID Pessoa: 693492 | Participações: 574
ID Pessoa: 518153 | Participações: 550
ID Pessoa: 532585 | Participações: 531


In [14]:
print("--- 4.1 Vantagem: Recuperar Produção e Equipe em uma ÚNICA leitura ---")
import time

start_time = time.time()
# No SQL, isso exigiria um JOIN entre Produção e Equipe. 
# Aqui, pegamos o documento inteiro de uma vez.
exemplo_vantagem = db.producoes_com_participantes.find_one({"id_producao": 12345}) 
end_time = time.time()

print(f"Título: {exemplo_vantagem['titulo']}")
print(f"Equipe Completa: {exemplo_vantagem['equipe']}")
print(f"⏱️ Tempo de resposta: {(end_time - start_time) * 1000:.4f} ms")

--- 4.1 Vantagem: Recuperar Produção e Equipe em uma ÚNICA leitura ---
Título: AI: A Portrait of David
Equipe Completa: [{'_id': ObjectId('69cc526dd81bd2a577c5e38a'), 'id_producao': 12345, 'id_pessoa': 47879, 'papel': 'Ukulele player'}]
⏱️ Tempo de resposta: 90.2772 ms


In [16]:
import sys
!{sys.executable} -m pip install sqlalchemy psycopg2-binary

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------- ----- 1.8/2.1 MB 20.2 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 17.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 31.4 MB/s  0:00:00

   ---------------------------------------- 0/3 [psycopg2-binary]
   ------------- -------------------------- 1/3 [greenlet]
   ------------- -------------------------- 1/3 [greenlet]
   ------------- -------------------------- 1/3 [greenlet]
   ------------- -------------------------- 1/3 [greenlet]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   --------------------------

In [21]:
import sys
!{sys.executable} -m pip install pg8000


   ---------------------------------------- 0/3 [asn1crypto]
   ---------------------------------------- 0/3 [asn1crypto]
   ---------------------------------------- 0/3 [asn1crypto]
   -------------------------- ------------- 2/3 [pg8000]
   ---------------------------------------- 3/3 [pg8000]



In [27]:
import pandas as pd
from sqlalchemy import create_engine

# As credenciais que estão no seu print do DBeaver
usuario = "postgres"
senha = "Caught04@" # Coloque sua senha aqui
host = "127.0.0.1"
porta = "5433" # Alterado para 5433 conforme seu print
banco = "postgres"

# Usando o driver pg8000 que instalamos, mas na porta 5433
url_conexao = f'postgresql+pg8000://{usuario}:{senha}@{host}:{porta}/{banco}'

try:
    engine = create_engine(url_conexao)
    print(f"--- 📡 Testando conexão na porta {porta} ---")

    # Amostra pequena para teste rápido
    df_prod = pd.read_parquet('Data/producao.parquet').head(10000)
    
    print("⏳ Tentando enviar dados para o PostgreSQL...")
    df_prod.to_sql('producao_relacional', engine, if_exists='replace', index=False)
    
    print("✅ FINALMENTE FUNCIONOU! O erro era a porta.")
    print("Pode conferir no DBeaver dentro do banco 'postgres'.")

except Exception as e:
    print(f"❌ Erro: {e}")

--- 📡 Testando conexão na porta 5433 ---
⏳ Tentando enviar dados para o PostgreSQL...
❌ Erro: (pg8000.exceptions.InterfaceError) Can't create a connection to host @127.0.0.1 and port 5433 (timeout is None and source_address is None).
(Background on this error at: https://sqlalche.me/e/20/rvf5)


In [7]:
import pandas as pd

# Carrega os dados do seu arquivo original
df_prod = pd.read_parquet('Data/equipe.parquet')

# Se quiser apenas uma amostra para testar rápido, use o .head(10000)
# df_prod = df_prod.head(10000) 

# Salva como CSV com delimitador de ponto e vírgula
df_prod.to_csv('equipe_final.csv', index=False, sep=';', encoding='utf-8')

print("Feito! O arquivo 'equipe_final.csv' foi criado.")

Feito! O arquivo 'equipe_final.csv' foi criado.
